# 02.1 — Generative applications lab

Five things, in order:

1. Three clients against one Foundry resource, and why each exists
2. Model families: small vs large vs reasoning vs code
3. Structured outputs with an enforced JSON schema
4. Streaming, and getting usage back from a streamed response
5. Multimodal input, then evaluating the result for fabrication, relevance, and safety

**Prerequisites:** `.env` written, `gpt-4o-mini` deployed. `gpt-4o` for the image
section, `o4-mini` for the reasoning comparison — both degrade gracefully if absent.

**Cost:** cents. Nothing here bills by the hour.

## 1. Connect

Same three lines every lab starts with.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client, ask, show_usage

print("project   :", cfg["AZURE_AI_PROJECT_ENDPOINT"])
print("inference :", cfg["AZURE_OPENAI_ENDPOINT"])
print("mini      :", cfg["MODEL_MINI"])
print("chat      :", cfg.get("MODEL_CHAT"))
print("reasoning :", cfg.get("MODEL_REASONING"))

## 2. Three clients, one resource

The exam contrasts these constantly. Run the cell and read what each one *cannot*
do as much as what it can.

| Client | Package | Endpoint | Owns |
|---|---|---|---|
| `AIProjectClient` | `azure-ai-projects` | project | deployments, connections, agents, evaluations, indexes |
| `AzureOpenAI` | `openai` | inference | chat, embeddings, images, structured outputs, streaming |
| `ChatCompletionsClient` | `azure-ai-inference` | inference | provider-neutral chat across OpenAI, Phi, Llama, Mistral |

In [ ]:
project = project_client()

# (a) AIProjectClient — the control plane. Only it knows what exists.
print("deployments visible to AIProjectClient:")
for d in project.deployments.list():
    print(f"  {d.name:<28} {getattr(d, 'model_name', '')} {getattr(d, 'type', '')}")

In [ ]:
# (b) AzureOpenAI — the data plane for OpenAI-family models.
#     Two ways to get one. They are equivalent; the second saves you deriving
#     the inference endpoint from the project endpoint by hand.
oai_manual = chat_client()                       # built in scripts/ai103.py
oai_from_project = project.get_openai_client()   # convenience on AIProjectClient

r = oai_from_project.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=[{"role": "user", "content": "Reply with the single word: connected"}],
)
print("get_openai_client() ->", r.choices[0].message.content)

In [ ]:
# (c) ChatCompletionsClient — the provider-neutral surface.
#     Identical code would work against Phi-4 or Llama with only the model name changed.
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage

inference = ChatCompletionsClient(
    endpoint=f"{cfg['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/deployments/{cfg['MODEL_MINI']}",
    credential=credential(),
    credential_scopes=["https://cognitiveservices.azure.com/.default"],
    api_version=cfg.get("AZURE_OPENAI_API_VERSION", "2025-04-01-preview"),
)

resp = inference.complete(
    messages=[
        SystemMessage(content="You are terse."),
        UserMessage(content="Name one advantage of a provider-neutral inference API."),
    ],
    max_tokens=60,
)
print(resp.choices[0].message.content)
print("\nusage:", resp.usage.prompt_tokens, "+", resp.usage.completion_tokens)
inference.close()

> **Exam note.** *"Which client creates an agent?"* → `AIProjectClient`, because
> agents belong to the project. *"Which client generates an embedding?"* →
> `AzureOpenAI` (or `EmbeddingsClient` from `azure-ai-inference`). The project
> client has no `.chat` and no `.embeddings`; it hands you a client that does.

## 3. Model families

Four categories from the study guide, one task each. The point is not that the big
model wins — it is to see where the small model is *good enough*, because that is
the decision the exam actually tests.

In [ ]:
import time

TICKET = (
    "Call from Dana Whitfield re invoice 88213. She is furious, says the unit "
    "arrived cracked, and wants a full refund credited by Friday or she escalates."
)


def timed(deployment, messages, **kw):
    """Call a deployment and report latency and tokens alongside the text."""
    t0 = time.perf_counter()
    r = chat_client().chat.completions.create(model=deployment, messages=messages, **kw)
    dt = time.perf_counter() - t0
    u = r.usage
    reasoning = 0
    details = getattr(u, "completion_tokens_details", None)
    if details is not None:
        reasoning = getattr(details, "reasoning_tokens", 0) or 0
    print(f"--- {deployment}  {dt:.2f}s  in={u.prompt_tokens} out={u.completion_tokens} reasoning={reasoning}")
    print(r.choices[0].message.content)
    print()
    return r


extract = [
    {"role": "system", "content": "Extract customer, invoice, sentiment, and deadline. Be terse."},
    {"role": "user", "content": TICKET},
]

# Small model — the default for extraction, classification, routing.
timed(cfg["MODEL_MINI"], extract, temperature=0)

# Large model — same task, more money. Compare the answers, not the vibes.
if cfg.get("MODEL_CHAT"):
    timed(cfg["MODEL_CHAT"], extract, temperature=0)

For extraction the two answers are usually indistinguishable while the large model
costs roughly ten times more per token. Now a task where the difference is real: a
constraint-satisfaction puzzle that needs multi-step reasoning.

In [ ]:
PUZZLE = (
    "A warehouse ships 3 crates. Crate A weighs twice crate B. Crate C weighs 12kg "
    "less than A. Total is 108kg. The truck's axle limit means no single crate may "
    "exceed 50kg. Give each weight and state whether the load is legal. "
    "Answer with the numbers only, then LEGAL or ILLEGAL."
)
puzzle_msgs = [{"role": "user", "content": PUZZLE}]

timed(cfg["MODEL_MINI"], puzzle_msgs, temperature=0)

# Reasoning model. Note the different parameter name and the absence of temperature.
if cfg.get("MODEL_REASONING"):
    try:
        timed(cfg["MODEL_REASONING"], puzzle_msgs, max_completion_tokens=2000)
    except Exception as e:
        print("reasoning model unavailable:", type(e).__name__, str(e)[:200])
else:
    print("MODEL_REASONING not set — skipping")

Read the `reasoning=` number on the reasoning-model line. Those are tokens you are
billed for and never see. They routinely exceed the visible output by 10x, which is
why "use a reasoning model" is a costed decision and not a free upgrade.

Also note what you could *not* pass: `temperature`. Reasoning models reject or
ignore it, along with `top_p` and the penalties, and they use
`max_completion_tokens` instead of `max_tokens`. Unit 02.5 has the full table.

In [ ]:
# "Code model" is a use case, not a deployment type. There is nothing special to
# deploy — you pick the chat model that codes best and give it a code-shaped prompt.
code_answer = ask(
    "Write a Python function `chunk(text, size, overlap)` that splits text into "
    "overlapping character windows. Return only code, no prose, no markdown fence.",
    system="You are a senior Python engineer. Output runnable code only.",
    temperature=0,
)
print(code_answer)

## 4. Structured outputs

Three escalating guarantees. Run all three and watch the third one become the only
thing you would put in production.

In [ ]:
import json

client = chat_client()
prompt = [
    {"role": "system", "content": "Extract a support ticket record."},
    {"role": "user", "content": TICKET},
]

# Level 1 — prompt only. Works until it does not.
loose = client.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=prompt + [{"role": "user", "content": "Reply in JSON."}],
    temperature=0,
)
print("LEVEL 1 (prompt only):")
print(repr(loose.choices[0].message.content[:200]))

In [ ]:
# Level 2 — JSON mode. Syntactically valid JSON. Field names still up to the model.
json_mode = client.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=prompt,
    response_format={"type": "json_object"},
    temperature=0,
)
parsed = json.loads(json_mode.choices[0].message.content)
print("LEVEL 2 (json_object) keys:", list(parsed.keys()))

In [ ]:
# Level 3 — structured outputs. The schema is enforced by constrained decoding.
#
# strict:true has three hard requirements the exam likes:
#   * additionalProperties: false on EVERY object, including nested ones
#   * every property listed in "required"
#   * optional fields expressed as a union with null, never by omission
TICKET_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "customer_name": {"type": "string"},
        "invoice_id": {"type": ["string", "null"]},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "severity": {"type": "integer", "description": "1 lowest, 5 highest"},
        "requested_action": {"type": "string"},
        "deadline": {"type": ["string", "null"]},
    },
    "required": [
        "customer_name", "invoice_id", "sentiment",
        "severity", "requested_action", "deadline",
    ],
}

strict = client.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=prompt,
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "ticket", "strict": True, "schema": TICKET_SCHEMA},
    },
    temperature=0,
)

msg = strict.choices[0].message
if getattr(msg, "refusal", None):
    print("model refused:", msg.refusal)   # always check this before parsing
else:
    record = json.loads(msg.content)
    print("LEVEL 3 (json_schema, strict):")
    print(json.dumps(record, indent=2))
    assert record["sentiment"] in {"positive", "neutral", "negative"}
    assert isinstance(record["severity"], int)
    print("\nschema conformance asserted")

> **Exam note.** JSON mode guarantees *parseable*, not *correct*. Only
> `json_schema` with `strict: true` guarantees the fields, types, and enum values
> you asked for. And always check `message.refusal` — a refused response has
> `content = None`, so `json.loads` would raise.

## 5. Streaming

Streaming does not make generation faster. It lowers **time to first token**, which
is what a user experiences as speed. Cost is identical.

In [ ]:
t0 = time.perf_counter()
first_token_at = None
usage = None

stream = client.chat.completions.create(
    model=cfg["MODEL_MINI"],
    messages=[{"role": "user", "content": "In three sentences, explain what a vector index is."}],
    stream=True,
    stream_options={"include_usage": True},   # without this, usage is None
)

for chunk in stream:
    if chunk.usage:                # arrives in a final chunk with empty choices
        usage = chunk.usage
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.perf_counter() - t0
        print(delta, end="")

total = time.perf_counter() - t0
print(f"\n\ntime to first token : {first_token_at:.2f}s")
print(f"total               : {total:.2f}s")
print(f"usage               : {usage}")

## 6. Multimodal input

Images ride in the `content` array as `image_url` parts. Private images go as a
`data:` URI containing base64 bytes — there is no upload step for chat completions.

This section uses **`MODEL_CHAT`** (`gpt-4o`) rather than the usual mini, because
vision needs a multimodal deployment. Image tokens are billed as **prompt** tokens,
and `detail="high"` tiles the image into many more of them than `detail="low"`.

In [ ]:
import base64, io

# Generate a tiny chart locally so the lab needs no external asset or network fetch.
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axis = plt.subplots(figsize=(4, 2.5))
    axis.bar(["Jan", "Feb", "Mar", "Apr"], [12, 19, 7, 23])
    axis.set_title("Support escalations")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=80, bbox_inches="tight")
    plt.close(fig)
    png_b64 = base64.b64encode(buf.getvalue()).decode()
    print(f"chart generated, {len(png_b64)} base64 chars")
except ImportError:
    png_b64 = None
    print("matplotlib not installed — skip the next cell or pip install matplotlib")

In [ ]:
vision_model = cfg.get("MODEL_CHAT")

if png_b64 and vision_model:
    for detail in ("low", "high"):
        vision = client.chat.completions.create(
            model=vision_model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": "Which month is highest, and by how much over the lowest?"},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{png_b64}",
                            "detail": detail,
                        },
                    },
                ],
            }],
            temperature=0,
            max_tokens=120,
        )
        print(f"detail={detail:<5} prompt_tokens={vision.usage.prompt_tokens}")
        print("  ", vision.choices[0].message.content.replace("\n", " ")[:220])
else:
    print("skipped — need matplotlib and a multimodal deployment (MODEL_CHAT)")

Compare the two `prompt_tokens` values. `detail="low"` is a flat, cheap cost;
`detail="high"` tiles the image and can multiply it several times over. For
thumbnails, logos, and simple charts, `"low"` is usually correct.

## 7. Evaluate: fabrication, relevance, quality, safety

The study-guide bullet names four things. They map onto three evaluator families
with genuinely different plumbing:

| Concern | Evaluator | Configured with | Score |
|---|---|---|---|
| Fabrication | `GroundednessEvaluator` | `model_config` (LLM-as-judge) | 1–5, higher better |
| Relevance / quality | `RelevanceEvaluator`, `CoherenceEvaluator`, `FluencyEvaluator` | `model_config` | 1–5, higher better |
| Ground-truth overlap | `F1ScoreEvaluator` | nothing — pure maths, free | 0–1 |
| Safety | `ViolenceEvaluator`, `HateUnfairnessEvaluator`, … | `azure_ai_project` + `credential` | 0–7 severity, **lower better** |

First, manufacture a grounded answer and a fabricated one so the scores have
something to separate.

In [ ]:
CONTEXT = (
    "Contoso returns policy: unopened items may be returned within 30 days for a "
    "full refund. Opened items receive store credit only. Refunds are issued to the "
    "original payment method within 5 business days of receipt."
)
QUERY = "I opened the box. Can I get my money back, and how long does it take?"

grounded = ask(
    f"Answer using ONLY this policy. If it is not stated, say so.\n\n{CONTEXT}\n\nQ: {QUERY}",
    temperature=0,
)

# A deliberately fabricated answer, so you can see groundedness actually move.
fabricated = (
    "Yes. Opened items qualify for a full cash refund within 90 days, and Contoso "
    "also reimburses return shipping and adds a 10% goodwill credit. Refunds clear "
    "the same day."
)

print("GROUNDED:\n", grounded, "\n")
print("FABRICATED:\n", fabricated)

In [ ]:
from azure.ai.evaluation import (
    GroundednessEvaluator,
    RelevanceEvaluator,
    CoherenceEvaluator,
    F1ScoreEvaluator,
)

# AI-assisted evaluators are LLM-as-judge: they need a deployment and cost tokens.
model_config = {
    "azure_endpoint": cfg["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": cfg["MODEL_MINI"],
    "api_version": cfg.get("AZURE_OPENAI_API_VERSION", "2025-04-01-preview"),
}

groundedness = GroundednessEvaluator(model_config)
relevance = RelevanceEvaluator(model_config)
coherence = CoherenceEvaluator(model_config)
f1 = F1ScoreEvaluator()   # no model_config — pure token overlap, costs nothing

for label, answer in (("grounded", grounded), ("fabricated", fabricated)):
    g = groundedness(query=QUERY, response=answer, context=CONTEXT)
    r = relevance(query=QUERY, response=answer)
    c = coherence(query=QUERY, response=answer)
    print(f"{label:<11} groundedness={g['groundedness']}  relevance={r['relevance']}  coherence={c['coherence']}")
    print(f"            reason: {str(g.get('groundedness_reason', ''))[:180]}")

Groundedness collapses on the fabricated answer while **relevance and coherence stay
high** — it is a fluent, on-topic, confident lie. That is exactly why you cannot ship
on relevance alone, and it is the reason groundedness is a separate metric.

`GroundednessProEvaluator` (**preview**) instead calls the Azure AI Content Safety
groundedness-detection service and returns the *ungrounded spans*, so you can point
at the sentence rather than just score the answer. It takes `azure_ai_project` and
`credential`, not `model_config`.

In [ ]:
# Safety evaluators are structurally different: they call a Microsoft-hosted safety
# service, not your deployment. Note the arguments, and note that the scale inverts
# — 0-7 severity where LOWER is better.
from azure.ai.evaluation import ViolenceEvaluator, HateUnfairnessEvaluator

try:
    violence = ViolenceEvaluator(
        azure_ai_project=cfg["AZURE_AI_PROJECT_ENDPOINT"], credential=credential()
    )
    hate = HateUnfairnessEvaluator(
        azure_ai_project=cfg["AZURE_AI_PROJECT_ENDPOINT"], credential=credential()
    )
    v = violence(query=QUERY, response=grounded)
    h = hate(query=QUERY, response=grounded)
    print("violence      :", v)
    print("hate/unfair   :", h)
except Exception as e:
    print("safety evaluators unavailable in this region/project:")
    print(" ", type(e).__name__, str(e)[:250])

In [ ]:
# Batch evaluation over a dataset — the shape the portal uses and the exam describes.
import json, tempfile, os
from azure.ai.evaluation import evaluate

rows = [
    {"query": QUERY, "response": grounded, "context": CONTEXT, "ground_truth": "Store credit only; 5 business days."},
    {"query": QUERY, "response": fabricated, "context": CONTEXT, "ground_truth": "Store credit only; 5 business days."},
]

dataset_path = os.path.join(tempfile.gettempdir(), "ai103_eval_rows.jsonl")
with open(dataset_path, "w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row) + "\n")

result = evaluate(
    data=dataset_path,
    evaluators={"groundedness": groundedness, "relevance": relevance, "f1": f1},
    # Add azure_ai_project=cfg["AZURE_AI_PROJECT_ENDPOINT"] to publish the run to
    # the portal's Evaluation blade instead of keeping it local.
)

print("aggregate metrics:")
for k, v in result["metrics"].items():
    print(f"  {k:<40} {v}")

> **Exam note.** Quality evaluators are 1–5, higher better. Safety evaluators are
> 0–7 severity, lower better. Quality evaluators need `model_config`; safety
> evaluators need `azure_ai_project` + `credential`. `F1`/`BLEU`/`ROUGE` need
> neither and cost nothing, but they require `ground_truth`.

## 8. Cleanup

This lab created no agents, indexes, or deployments — only a temp file. Later units
create real, billable things and their cleanup cells matter far more.

In [ ]:
import os

for path in [dataset_path]:
    try:
        os.remove(path)
        print("removed", path)
    except (OSError, NameError):
        pass

print("nothing billable was created by this lab")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Break strict mode.** Remove `"additionalProperties": False` from
   `TICKET_SCHEMA` and resend. Record the exact error. Then remove one field from
   `required` instead and record that error. What do the two messages tell you
   about how strict mode is validated — client-side or server-side?
2. **Cost the image.** Call the vision model on the same chart at `detail="low"`
   and `detail="high"`, and compute the ratio of prompt tokens. At $2.50 per
   million input tokens, how many `"high"` calls buy you $1 of spend?
3. **A judge that disagrees with itself.** Run `RelevanceEvaluator` three times on
   the *same* query/response pair. Do you get the same score each time? What does
   that tell you about using a single evaluator run as a release gate?
4. **Portability.** Rewrite the `ChatCompletionsClient` call to use
   `EmbeddingsClient` from `azure-ai-inference` against `MODEL_EMBEDDING`, and
   confirm the vector length matches what `ai103.embed()` returns.

In [ ]:
# Your work here.